In [ ]:
# FASE 5 — EVALUATION | CardioRisk · IBM Data Science · CRISP-DM
!pip install -q kagglehub scikit-learn imbalanced-learn matplotlib shap xgboost
import os, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
warnings.filterwarnings('ignore')
matplotlib.rcParams['figure.facecolor'] = '#0a0f1a'
matplotlib.rcParams['axes.facecolor']   = '#0d1526'
matplotlib.rcParams['text.color']       = '#e2e8f0'
matplotlib.rcParams['axes.labelcolor']  = '#e2e8f0'
matplotlib.rcParams['xtick.color']      = '#7a8fa8'
matplotlib.rcParams['ytick.color']      = '#7a8fa8'
matplotlib.rcParams['axes.edgecolor']   = '#1a2c3d'
matplotlib.rcParams['grid.color']       = '#1a2c3d'
matplotlib.rcParams['savefig.facecolor']= '#0a0f1a'
print('✓ Librerías cargadas')

In [ ]:
# BLOQUE 1 — PIPELINE COMPLETO (autocontenido)
import kagglehub
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import GradientBoostingClassifier
try:
    path = kagglehub.dataset_download('jocelyndumlao/cardiovascular-disease-dataset')
    csv_path = next(f for f in [os.path.join(r,f) for r,_,fs in os.walk(path) for f in fs] if f.endswith('.csv'))
    df = pd.read_csv(csv_path)
except:
    df = pd.read_csv('/content/cardiovascular_disease_dataset.csv')
df.columns = df.columns.str.lower().str.strip()
df = df.dropna()
CATEGORICAL  = ['gender', 'chestpain', 'restingrelectro']
TARGET       = 'target'
df_enc       = pd.get_dummies(df, columns=CATEGORICAL, drop_first=True)
feature_cols = [c for c in df_enc.columns if c != TARGET and c != 'patientid']
print(f'✓ patientid excluido. Features: {len(feature_cols)}')
X = df_enc[feature_cols]
y = df_enc[TARGET]
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1765, random_state=42, stratify=y_temp)
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)
sm = SMOTE(random_state=42)
X_train_sm, y_train_sm = sm.fit_resample(X_train_sc, y_train)
best_model = GradientBoostingClassifier(learning_rate=0.1, max_depth=5, n_estimators=200, random_state=42)
best_model.fit(X_train_sm, y_train_sm)
y_pred = best_model.predict(X_test_sc)
y_prob = best_model.predict_proba(X_test_sc)[:, 1]
print(f'Evaluación sobre TEST SET ({len(y_test)} pacientes no vistos)')

In [ ]:
# BLOQUE 2 — MÉTRICAS COMPLETAS
from sklearn.metrics import (recall_score, precision_score, f1_score, roc_auc_score, accuracy_score, confusion_matrix, classification_report)
recall    = recall_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred)
auc_val   = roc_auc_score(y_test, y_prob)
accuracy  = accuracy_score(y_test, y_pred)
cm        = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
especificidad  = tn / (tn + fp)
print('═' * 50)
print(f'  Recall (Sensibilidad) : {recall:.4f}  ← MÉTRICA PRIMARIA')
print(f'  Precision             : {precision:.4f}')
print(f'  F1-Score              : {f1:.4f}')
print(f'  AUC-ROC               : {auc_val:.4f}')
print(f'  Accuracy              : {accuracy:.4f}')
print(f'  Especificidad         : {especificidad:.4f}')
print('─' * 50)
print(f'  TP={tp}  TN={tn}  FP={fp}  FN={fn}')
print('═' * 50)
print(f'{classification_report(y_test, y_pred, target_names=["Bajo Riesgo","Alto Riesgo"])}')

In [ ]:
# BLOQUE 3-5 — MATRIZ CONFUSIÓN, ROC, SHAP
from sklearn.metrics import roc_curve, precision_recall_curve, average_precision_score
import shap
fpr, tpr, _ = roc_curve(y_test, y_prob)
fig, ax = plt.subplots(figsize=(7,5))
ax.plot(fpr, tpr, color='#00f2fe', lw=2, label=f'AUC={auc_val:.4f}')
ax.plot([0,1],[0,1],'--',color='#3a5570',lw=1)
ax.fill_between(fpr, tpr, alpha=.07, color='#00f2fe')
ax.set_xlabel('FPR'); ax.set_ylabel('TPR (Recall)')
ax.set_title('Curva ROC — Test Set'); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig('/content/f5_roc.png', dpi=150, bbox_inches='tight'); plt.show()
kpi = '✓ KPI ALCANZADO' if recall >= 0.80 else f'⚠ KPI objetivo: 0.80 | Obtenido: {recall:.3f}'
print(f'  {kpi}')
print('→ FASE 5 COMPLETA')